# Suite PYR — The Python RegExp dialect

The regular expressions of Python's `re` module, a grammar of their own (the Library Reference, `re`, "Regular
Expression Syntax"): the dialect `Expressions.Python.RegExp` of the Python family, its kinds the constructs `re`'s
syntax lists, named as regexpp's where ECMAScript has them too, its `Term` of them alone, its holder, its checks, as
`re`'s parser makes them, its editions, its printing, its evaluation, which refuses each kind by name, and its
completeness against the codes `re`'s parser writes (the dialects proposal, Retrofit's design, Python, and its Open, 1,
"(b)").

In [ ]:
import { readFileSync } from "node:fs";

import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns";
import * as PythonRegExp from "@mbse/expressions/Python/RegExp";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, form_equal, map, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [P, PV, PText] = [Python3.Expressions, Python3.Evaluators, Python3.Text];
const [R, RText] = [PythonRegExp.Expressions, PythonRegExp.Text];
const [PT, PP] = [PythonTypes.Expressions, PythonPatterns.Expressions];

const store: any = new Builders.Store(); // the framework, the standard dialects Python3's complex numbers use, and the Python family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
  PythonTypes.install, PythonPatterns.install, PythonRegExp.install]) {
  store.transaction(install);
}
store.transaction(PythonRegExp.define); // the versions the RegExp dialect covers
const Term: any = store.registered("Expressions.Framework.Term");

/** The holder of the term `spec` writes, `dialect`'s own (the RegExp dialect's unless given), written in a
 * transaction. */
function held(spec: unknown, dialect: F.Dialect = R.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

/** A quantifier of `element`, its bounds and its options where given. */
function repeat(element: unknown, least: bigint | null = null, most: bigint | null = null,
  options: { greedy?: boolean; possessive?: boolean } = {}): unknown {
  return (u: any) => u.quantifier((q: any) => {
    if (least !== null) q.min(least);
    if (most !== null) q.max(most);
    if (options.greedy !== undefined) q.greedy(options.greedy);
    if (options.possessive !== undefined) q.possessive(options.possessive);
    return element !== null ? q.element(element) : q;
  });
}

const regexp = (spec: unknown): any => F.term(held(spec)); // a term of the RegExp dialect, held by its holder
const char = (value: bigint, text: string | null = null) => (u: any) => u.character((c: any) => (text === null ? c.value(value) : c.value(value).text(text)));
const seq = (...elements: unknown[]) => (u: any) => u.alternative((a: any) => a.elements(elements)); // an alternative
const alts = (...alternatives: unknown[]) => (u: any) => u.pattern((p: any) => p.alternatives(alternatives)); // a pattern
const group = (...alternatives: unknown[]) => (u: any) => u.group((g: any) => g.alternatives(alternatives)); // `(?:...)`
const capture = (...alternatives: unknown[]) => (u: any) => u.capturinggroup((g: any) => g.alternatives(alternatives)); // `(...)`
const named = (name: string, ...alternatives: unknown[]) => (u: any) => u.capturinggroup((g: any) => g.name(name).alternatives(alternatives)); // `(?P<name>...)`
const atomic = (...alternatives: unknown[]) => (u: any) => u.atomicgroup((g: any) => g.alternatives(alternatives)); // `(?>...)`
const klass = (...elements: unknown[]) => (u: any) => u.characterclass((c: any) => c.elements(elements)); // `[...]`
const klassNot = (...elements: unknown[]) => (u: any) => u.characterclass((c: any) => c.negate(true).elements(elements)); // `[^...]`
const span = (low: unknown, high: unknown) => (u: any) => u.characterclassrange((r: any) => r.min(low).max(high)); // `a-z`
const charset = (kind: string, negate: boolean | null = null) => (u: any) => u.characterset( // `.`, `\d`, `\D`
  (s: any) => (negate === null ? s.kind(kind) : s.kind(kind).negate(negate)));
const check = (kind: string, ...alternatives: unknown[]) => (u: any) => u.assertion((a: any) => a.kind(kind).alternatives(alternatives)); // `^`, `(?=...)`
const checkNot = (kind: string, ...alternatives: unknown[]) => (u: any) => u.assertion( // `\B`, `(?!...)`
  (a: any) => a.kind(kind).negate(true).alternatives(alternatives));
const ref = (key: bigint | string) => (u: any) => u.backreference((b: any) => (typeof key === "bigint" ? b.number(key) : b.name(key)));
const cond = (key: bigint | string, yes: unknown, no: unknown = null) => (u: any) => u.conditional((c: any) => { // `(?(1)yes|no)`
  const keyed = typeof key === "bigint" ? c.number(key) : c.name(key);
  return no === null ? keyed.yes(yes) : keyed.yes(yes).no(no);
});
const flags = (add: string, remove = "", ...alternatives: unknown[]) => (u: any) => u.flags((f: any) => { // `(?i)`, `(?i-m:...)`
  const added = add !== "" ? f.add(add) : f;
  return (remove !== "" ? added.remove(remove) : added).alternatives(alternatives);
});
const comment = (text: string | null = null) => (u: any) => u.comment((c: any) => (text === null ? c : c.text(text))); // `(?#...)`
const [a, b] = [char(97n), char(98n)];

<a id="PYR-01"></a>
## PYR-01 · The RegExp dialect's union, its kinds' schemas named as regexpp's, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML; read by each dialect of its family

In [ ]:
// Its kinds: the constructs `re`'s syntax lists, named and shaped as regexpp's where ECMAScript has them too, then
// Python's own and its comment; its Term lists them and invalid syntax alone, and it has no helper of its own.
const union = (dialect: F.Dialect): string[] => store.registered(dialect.union).branches.map((b: any) => b.name); // its Term's branches
assert(same(R.TAGS, ["pattern", "alternative", "group", "capturinggroup", "quantifier", "characterclass", "characterclassrange",
  "characterset", "character", "assertion", "backreference", "conditional", "atomicgroup", "flags", "comment"]) && R.HELPERS.length === 0);
assert(same(union(R.DIALECT), [...R.TAGS, "error", "missing"]) && same([...R.DIALECT.kinds().keys()], [...R.TAGS, "error", "missing"]));
assert(!R.TAGS.some((tag) => [P.DIALECT, PT.DIALECT, PP.DIALECT].some((d) => union(d).includes(tag) || d.kinds().has(tag)))); // its tags its own
assert(R.DIALECT.name() === "Python RegExp" && R.DIALECT.tag === "python_regexp");
assert(R.DIALECT.path === "Python.RegExp" && R.DIALECT.union === "Expressions.Python.RegExp.Term");
const RegExpTerm: any = store.registered(R.DIALECT.union);
const kinds = new Map<string, any>(RegExpTerm.branches.map((b: any) => [b.name, b.schema]));
assert(same(R.TAGS.map((tag) => kinds.get(tag).name), ["Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier",
  "CharacterClass", "CharacterClassRange", "CharacterSet", "Character", "Assertion", "Backreference", "Conditional", "AtomicGroup",
  "Flags", "Comment"].map((kind) => `Expressions.Python.RegExp.${kind}`)));
assert(equal(new Map(R.TAGS.map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({ // each a value object, its properties regexpp's names
  pattern: ["alternatives"], alternative: ["elements"], group: ["alternatives"],
  capturinggroup: ["name", "alternatives"], quantifier: ["min", "max", "greedy", "possessive", "element"],
  characterclass: ["negate", "elements"], characterclassrange: ["min", "max"], characterset: ["kind", "negate"],
  character: ["value", "text"], assertion: ["kind", "negate", "alternatives"], backreference: ["number", "name"],
  conditional: ["number", "name", "yes", "no"], atomicgroup: ["alternatives"],
  flags: ["add", "remove", "alternatives"], comment: ["text"] })) && !R.TAGS.some((t) => kinds.get(t).ref));
assert(([["pattern", "alternatives"], ["alternative", "elements"], ["group", "alternatives"], ["capturinggroup", "alternatives"],
  ["characterclass", "elements"], ["assertion", "alternatives"], ["atomicgroup", "alternatives"], ["flags", "alternatives"]] as
  [string, string][]).every(([tag, name]) => kinds.get(tag).properties.get(name).schema.element === RegExpTerm));
assert(([["quantifier", "element"], ["characterclassrange", "min"], ["characterclassrange", "max"], ["conditional", "yes"],
  ["conditional", "no"]] as [string, string][]).every(([tag, name]) => kinds.get(tag).properties.get(name).schema === RegExpTerm));
assert(equal(F._parts(kinds.get("quantifier")), [["element"], [], ["min", "max", "greedy", "possessive"]]));
assert(equal(F._parts(kinds.get("conditional")), [["yes", "no"], [], ["number", "name"]]));
assert(equal(F._parts(kinds.get("flags")), [[], ["alternatives"], ["add", "remove"]]) && equal(F._parts(kinds.get("character")), [[], [], ["value", "text"]]));
// No position of its family's other dialects holds its Term: a pattern is a string in Python3, which no reader guesses
// is a regular expression (PRI-025); mbse-programs, or a constraint's author, holds one through its holder. Its Term is a
// branch of the framework's, its dialect a singleton.
const holds = (schema: any): boolean => [...schema.properties.values()].some((p: any) => p.schema === RegExpTerm || p.schema.element === RegExpTerm);
assert(![P.DIALECT, PT.DIALECT, PP.DIALECT].some((d) => store.registered(d.union).branches.some((b: any) => holds(b.schema))));
assert(Term.branches.find((b: any) => b.name === "python_regexp").schema === RegExpTerm);
assert(store.singleton("Expressions.Python.RegExp") === store.Instances.Expressions.Python.RegExp);
// Each dialect of the family reads its kinds, and it reads theirs: each reads the other three.
assert(same(R.DIALECT.reads(), [P.DIALECT, PT.DIALECT, PP.DIALECT]));
assert([P.DIALECT, PT.DIALECT, PP.DIALECT].every((d) => d.reads().includes(R.DIALECT)));
assert(P.DIALECT.kindNamed("quantifier") === R.DIALECT.kinds().get("quantifier") && R.DIALECT.kindNamed("binop") === P.DIALECT.kinds().get("binop"));
// Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
// validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
const Holder: any = store.registered("Expressions.Python.RegExp.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === RegExpTerm && Holder.validate().length === 0 && same([...Holder.adjacencies.keys()], ["abi"]));
const every = alts( // (?P<n>a\d+)(?P=n)|[^a-z_]\b
  seq(named("n", seq(a, repeat(charset("digit"), 1n))), ref("n")),
  seq(klassNot(span(a, char(122n)), char(95n)), check("word")));
const [holder, linked] = store.transaction(() => {
  const holder = store.Expressions.Python.RegExp.Expression((e: any) => e.root(R.DIALECT.spec(every)));
  const linked = store.Schemas.OfExpression((s: any) => s.name("pyr.every").expression((x: any) => x.insert((e: any) => e.expression(holder))));
  return [holder, linked];
});
assert(Validators.Validate(store)(Holder, holder).length === 0 && R.DIALECT.validate(holder).length === 0);
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
const SOURCE = "(?P<n>a\\d+)(?P=n)|[^a-z_]\\b";
assert(F.dialectOf(F.term(holder)) === R.DIALECT.union && RText.ToText(holder) === SOURCE);
for (const [write, read] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
  assert(F.same(read(Holder, write(Holder, holder)).root, holder.root));
  assert(F.same(read(RegExpTerm, write(RegExpTerm, F.term(holder))), F.term(holder)));
}
// The schema refuses a regular expression where a union that lacks it is held: Python3's holder's root.
raises(Error, () => store.transaction(() => store.Expressions.Python.Python3.Expression(
  (e: any) => e.root(R.DIALECT.make(new F.Form("pattern", {}, []))))), "'pattern' is not a member of a union value");
// Another family's term is refused, as before.
raises(ValueError, () => R.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1n, Patterns.Expressions.variable("a")),
  Patterns.Expressions.DIALECT))), "a Patterns let is not a Python RegExp term");

<a id="PYR-02"></a>
## PYR-02 · Each kind and its checks, as `re`'s parser makes them, with its messages: what each kind may hold, a choice among a few, a quantifier's bounds and element, a range's order, a reference's group, an inline flags' letters

In [ ]:
// Each kind's checks: what it may hold (an alternative where alternatives stand; no pattern, alternative or range
// among an alternative's elements; a character, a range or a set within a class; a character at either end of a range;
// an element a quantifier can repeat), a choice among a few (a set's and an assertion's kind), and its attributes (a
// quantifier's bounds and options, a reference's group, a group's name, an inline flags' letters, a comment's text),
// each with `re`'s parser's message where it has one. A backreference out of a pattern is not resolved (PYR-04).
const validate = (spec: unknown): string[] => R.DIALECT.validate(regexp(spec));
for (const [spec, problems] of [
  [alts(seq(a, group(seq(b), seq()))), []], [seq(), []],
  [alts(a), ["a pattern's alternative cannot be a character"]],
  [group(seq(a), b), ["a group's alternative cannot be a character"]],
  [named("x", a), ["a capturinggroup's alternative cannot be a character"]],
  [atomic(seq(a)), []], [atomic(alts(seq())), ["an atomicgroup's alternative cannot be a pattern"]],
  [seq(alts(seq()), seq(), span(a, b)), ["an alternative's element cannot be a pattern",
    "an alternative's element cannot be an alternative",
    "an alternative's element cannot be a characterclassrange"]],
  [klass(a, span(a, char(122n)), charset("digit", true)), []],
  [klass(group(seq(a)), check("start")), ["a characterclass's element cannot be a group",
    "a characterclass's element cannot be an assertion"]],
  [klass(charset("any")), ["a characterclass's element cannot be the characterset any"]],
  [klass(), ["a characterclass needs an element"]],
  [span(char(122n), a), ["bad character range z-a"]], [span(char(93n, "\\]"), char(65n)), ["bad character range \\]-A"]],
  [span(charset("digit"), a), ["a characterclassrange's min must be a character, got a characterset"]],
  [span(a, klass(b)), ["a characterclassrange's max must be a character, got a characterclass"]],
  [(u: any) => u.characterclassrange((r: any) => r.min(a)), ["a characterclassrange needs a max"]],
  [charset("any"), []], [charset("word", true), []], [charset("letter"), ["CharacterSet.kind cannot be 'letter'"]],
  [charset("any", true), ["the characterset any cannot be negated"]],
  [(u: any) => u.characterset((s: any) => s.negate(true)), ["a characterset needs a kind"]],
  [check("start"), []], [checkNot("word"), []], [check("lookahead", seq(a)), []],
  [checkNot("lookbehind", seq(a), seq(b)), []],
  [checkNot("end"), ["the assertion end cannot be negated"]],
  [check("lookahead"), ["the assertion lookahead needs an alternative"]],
  [check("stringend", seq(a)), ["the assertion stringend holds no alternatives"]],
  [check("lookahead", a), ["an assertion's alternative cannot be a character"]],
  [check("ahead"), ["Assertion.kind cannot be 'ahead'"]],
  [(u: any) => u.assertion((s: any) => s.negate(true)), ["an assertion needs a kind"]],
  [repeat(a, 0n), []], [repeat(a, 2n, 5n, { greedy: false }), []], [repeat(a, 1n, null, { possessive: true }), []],
  [repeat(a, 0n, null, { greedy: true, possessive: false }), []], [repeat(a, 3n, 3n), []],
  [repeat(a, -1n), ["a quantifier's min cannot be negative, got -1"]],
  [repeat(a, 2n, 1n), ["min repeat greater than max repeat"]],
  [repeat(a, 0n, null, { greedy: false, possessive: true }), ["a quantifier is lazy or possessive, not both"]],
  [repeat(repeat(a, 0n), 1n), ["multiple repeat"]],
  [repeat(check("start"), 0n), ["nothing to repeat"]], [repeat(check("word"), 0n), ["nothing to repeat"]],
  [repeat(check("lookahead", seq(a)), 0n), []], [repeat(comment("x"), 0n), ["nothing to repeat"]],
  [repeat(flags("i"), 0n), ["nothing to repeat"]], [repeat(flags("i", "", seq(a)), 0n), []],
  [repeat(seq(a), 0n), ["nothing to repeat"]],
  [repeat(group(seq(a)), 0n), []], [repeat(ref(1n), 1n), []], [repeat(cond(1n, seq(a)), 0n), []],
  [repeat(a), ["a quantifier needs a min"]], [repeat(null, 0n), ["a quantifier needs an element"]],
  [ref(1n), []], [ref("name"), []], [ref(0n), ["a backreference's number must be positive, got 0"]],
  [ref("1n"), ["a backreference's name must be an identifier, got '1n'"]],
  [(u: any) => u.backreference((r: any) => r.number(1n).name("n")), ["a backreference has a number or a name"]],
  [(u: any) => u.backreference((r: any) => r), ["a backreference has a number or a name"]],
  [cond(1n, seq(a)), []], [cond("n", seq(a), seq()), []],
  [cond(-2n, seq(a)), ["a conditional's number must be positive, got -2"]],
  [cond(1n, a), ["a conditional's yes must be an alternative, got a character"]],
  [cond(1n, seq(), b), ["a conditional's no must be an alternative, got a character"]],
  [(u: any) => u.conditional((c: any) => c.number(1n)), ["a conditional needs a yes"]],
  [capture(seq(a)), []], [named("a b", seq(a)), ["a capturinggroup's name must be an identifier, got 'a b'"]],
  [flags("i"), []], [flags("aimsx"), []], [flags("L"), []], [flags("", "i", seq(a)), []], [flags("u", "imsx", seq()), []],
  [flags(""), ["missing flag"]], [flags("q"), ["unknown flag"]], [flags("i", "q", seq()), ["unknown flag"]],
  [flags("au"), ["bad inline flags: flags 'a', 'u' and 'L' are incompatible"]],
  [flags("i", "a", seq(a)), ["bad inline flags: cannot turn off flags 'a', 'u' and 'L'"]],
  [flags("i", "m"), ["missing :"]],
  [flags("is", "s", seq(a)), ["bad inline flags: flag turned on and off"]],
  [comment("a note"), []], [comment(), []], [comment("a)b"), ["a comment's text cannot hold ')'"]],
] as [unknown, string[]][]) {
  assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}
// A kind's problems are reported where it nests, by its position.
assert(same(validate(alts(seq(a, repeat(b, 2n, 1n)))), ["argument 0: argument 1: min repeat greater than max repeat"]));
assert(R.problems(F.node("character", F.kindOf(regexp(a))[1])).length === 0);

<a id="PYR-03"></a>
## PYR-03 · A character: its code point and its spelling as written, checked against each other, and its default spelling outside a class and within one

In [ ]:
// A character is a domain representation (PRI-023): its value, a code point, beside its text, its spelling as written,
// which `spells` checks as `re`'s parser reads one: itself, a control character's escape, a hexadecimal, an octal or a
// named escape (its name's form alone), or a backslash before ASCII punctuation.
for (const [text, value, spelled] of [
  ["a", 97n, true], ["é", 0xE9n, true], ["😀", 0x1F600n, true], ["b", 97n, false], ["\\", 92n, false], ["ab", 97n, false],
  ["\\n", 10n, true], ["\\t", 9n, true], ["\\a", 7n, true], ["\\f", 12n, true], ["\\v", 11n, true], ["\\r", 13n, true],
  ["\\b", 8n, true], ["\\n", 13n, false], ["\\0", 0n, true], ["\\.", 46n, true], ["\\-", 45n, true], ["\\ ", 32n, true],
  ["\\d", 100n, false], ["\\q", 113n, false], ["\\1", 49n, false], ["\\é", 0xE9n, false],
  ["\\x41", 65n, true], ["\\x4", 4n, false], ["\\x4g", 0x4n, false], ["\\x42", 65n, false], ["\\u00e9", 0xE9n, true],
  ["\\u00E9", 0xE9n, true], ["\\U0001f600", 0x1F600n, true], ["\\U1f600", 0x1F600n, false],
  ["\\N{LATIN SMALL LETTER A}", 97n, true], ["\\N{}", 97n, false], ["\\N{A}B}", 97n, false], ["\\NA", 97n, false],
  ["\\N{EM DASH", 0x2014n, false], ["\\101", 65n, true], ["\\012", 10n, true], ["\\01", 1n, true], ["\\377", 255n, true],
  ["\\400", 256n, false], ["\\0123", 83n, false], ["\\18", 1n, false], ["\\101", 66n, false], ["x\\", 120n, false],
] as [string, bigint, boolean][]) {
  assert(R.spells(text, value) === spelled, `${text} ${value}`);
}
// Without a text, a character is written in its default spelling: printable ASCII as itself, escaped where it is
// special, outside a class (`re`'s special characters, a space and `#`, which verbose mode skips) or within one (a
// backslash, `]`, `^`, `-` and `[`); a control character by its escape or its code; any other as itself, but a
// surrogate, by its code. Each default spelling spells its value.
for (const [value, outside, within] of [
  [97n, "a", "a"], [46n, "\\.", "."], [93n, "\\]", "\\]"], [45n, "-", "\\-"], [94n, "\\^", "\\^"], [91n, "\\[", "\\["],
  [32n, "\\ ", " "], [35n, "\\#", "#"], [124n, "\\|", "|"], [92n, "\\\\", "\\\\"], [10n, "\\n", "\\n"], [7n, "\\a", "\\a"],
  [0n, "\\x00", "\\x00"], [8n, "\\x08", "\\x08"], [0x7Fn, "\\x7f", "\\x7f"], [0x85n, "\\x85", "\\x85"], [0xA0n, " ", " "],
  [0xE9n, "é", "é"], [0x2014n, "—", "—"], [0xD800n, "\\ud800", "\\ud800"], [0x1F600n, "😀", "😀"],
] as [bigint, string, string][]) {
  assert(R.spelling(value) === outside && R.spelling(value, true) === within, `${value} ${R.spelling(value)} ${R.spelling(value, true)}`);
  assert(R.spells(outside, value) && R.spells(within, value));
}
assert(R.spelling(0x110000n) === "\\U00110000"); // beyond Unicode, as validation refuses, by its code
// Validation checks a character's value is a code point, and its text, where it has one, spells it.
for (const [spec, problems] of [
  [char(97n), []], [char(97n, "a"), []], [char(97n, "\\x61"), []], [char(0x10FFFFn), []],
  [char(97n, "b"), ["a character's text 'b' does not spell U+0061"]],
  [char(0x1F600n, "\\u1f60"), ["a character's text '\\\\u1f60' does not spell U+1F600"]],
  [char(-1n), ["a character's value must be a code point, got -1"]],
  [char(0x110000n, "x"), ["a character's value must be a code point, got 1114112"]],
  [(u: any) => u.character((c: any) => c.text("a")), ["a character needs a value"]],
  [(u: any) => u.character((c: any) => c.value(true)), ["a character's value must be a int, got bool"]],
] as [unknown, string[]][]) {
  assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}

<a id="PYR-04"></a>
## PYR-04 · A pattern's groups and references, checked at its root; its forms and `make`; no kind binds a name, and a tree is read by each dialect of its family

In [ ]:
// At its root, a pattern's groups are numbered in the order they open, each name once, and each backreference and
// conditional refers to a group it has; a global flags stands only at its start, before anything but comments and other
// global flags (`re`'s parser's messages). Out of a pattern, nothing is resolved.
const ngroup = named("n", seq(b));
for (const [spec, problems] of [
  [alts(seq(capture(seq(a)), ngroup, ref(2n), ref("n"), cond(1n, seq(a), seq(b)), cond("n", seq()))), []],
  [alts(seq(named("n", seq(a)), ngroup)), ["redefinition of group name 'n' as group 2; was group 1"]],
  [alts(seq(ngroup, ref("m"), cond("m", seq()))), ["unknown group name 'm'", "unknown group name 'm'"]],
  [alts(seq(ngroup, ref(2n)), seq(cond(3n, seq()))), ["invalid group reference 2", "invalid group reference 3"]],
  [alts(seq(ngroup), seq(capture(seq(capture(seq(a)), ref(3n))))), []], // groups numbered across alternatives, nested
  [alts(seq(flags("i"), comment("c"), flags("m"), a)), []], [alts(seq(flags("x")), seq(a)), []],
  [alts(seq(a, flags("i"))), ["global flags not at the start of the expression"]],
  [alts(seq(), seq(flags("i"))), ["global flags not at the start of the expression"]],
  [alts(seq(group(seq(flags("i"))))), ["global flags not at the start of the expression"]],
  [alts((u: any) => u.missing((m: any) => m.kind("alternative")), seq(flags("i"))),
    ["global flags not at the start of the expression", "argument 0: missing alternative"]],
  [alts(seq(flags("i", "", seq(a)), a)), []], [alts(), []],
  [alts(seq((u: any) => u.backreference((r: any) => r.number(1n).name("n")), ref(0n))),
    ["argument 0: argument 0: a backreference has a number or a name",
      "argument 0: argument 1: a backreference's number must be positive, got 0"]],
] as [unknown, string[]][]) {
  assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}
// Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
// from which make writes it again, by the dialect whose union holds it.
assert(R.TAGS.every((tag) => R.DIALECT.kinds().get(tag)!.role === F.APPLICATION));
const samples = alts(
  seq(group(seq(a), seq()), named("n", seq(repeat(klass(span(a, char(122n)), charset("digit")), 1n, 3n, { greedy: false }))),
    checkNot("lookbehind", seq(b)), ref("n"), ref(1n), cond(1n, seq(a), seq(b)), atomic(seq(charset("any"))),
    repeat(b, 0n, null, { possessive: true }), check("start"), char(10n, "\\n"), comment("c")),
  seq(flags("i", "m", seq(a)), check("end")));
const tree = regexp(samples);
const walked = [...F.walk(tree)];
assert(same([...new Set(walked.map((node) => F.kindOf(node)[0]))].sort(), [...R.TAGS].sort()) && walked.length === 37);
for (const node of walked) {
  const made = F.term(held(R.DIALECT.make(F.form(node))));
  assert(F.same(made, node) && made !== node);
}
const quantified = F.form(F.form(F.form(F.form(tree).arguments[0]).arguments[1]).arguments[0]).arguments[0];
const formed = F.form(quantified);
assert(form_equal(formed, new F.Form("quantifier", { min: 1n, max: 3n, greedy: false }, formed.arguments)) && formed.arguments.length === 1);
assert(F.form(regexp(cond(1n, seq(a)))).arguments[1] === null && form_equal(F.form(regexp(flags("i"))), new F.Form("flags", { add: "i" }, [])));
// A named group's name is a label a backreference resolves, not a name the expression binds: no kind refines Let or has
// scopes, and a tree has no free names. Every dialect of the family reads it, validates it and prints it alike.
assert(!R.TAGS.some((tag) => F.isBinder(kinds.get(tag)) || R.DIALECT.kinds().get(tag)!.scopes !== null));
assert(SY.free(tree, R.DIALECT).size === 0 && F.nameOf(regexp(ngroup)) === "n");
assert(R.DIALECT.infer(tree) === D.Anything);
assert([R.DIALECT, P.DIALECT, PT.DIALECT, PP.DIALECT].every((dialect) => dialect.validate(tree).length === 0));
assert(RText.ToText === PText.ToText && PText.ToText(tree) === RText.ToText(tree));
assert(RText.ToText(tree) === "(?:a|)(?P<n>[a-z\\d]{1,3}?)(?<!b)(?P=n)\\1(?(1)a|b)(?>.)b*+^\\n(?#c)|(?i-m:a)$");

<a id="PYR-05"></a>
## PYR-05 · Invalid syntax where a regular expression's term stands: validated, printed as written, and refused by evaluation

In [ ]:
// Invalid syntax: the RegExp dialect's Term holds `error` and `missing`, wherever its terms stand; validation reports
// them, printers write them as written, and evaluation refuses them by the RegExp dialect's name.
const broken = regexp(alts(seq(a, (u: any) => u.error((e: any) => e.text("[a-").terms([R.DIALECT.spec(b)]))),
  (u: any) => u.missing((m: any) => m.kind("alternative"))));
assert(same(R.DIALECT.validate(broken), ["argument 0: argument 1: invalid syntax '[a-'", "argument 1: missing alternative"]));
assert(RText.ToText(broken) === "a[a-|");
assert(RText.ToText(regexp(seq(capture(seq(a)), ref(1n), (u: any) => u.missing((m: any) => m)))) === "(a)\\1"); // missing, written as nothing
assert(same(F.form(broken).arguments.map((x) => R.DIALECT.kind(x)[2].role), [F.APPLICATION, F.INVALID]));
const within = regexp(klass(a, (u: any) => u.error((e: any) => e.text("\\")), span((u: any) => u.missing((m: any) => m), b)));
assert(same(R.DIALECT.validate(within), ["argument 1: invalid syntax '\\\\'", "argument 2: min: missing a term"]));
assert(RText.ToText(within) === "[a\\-b]");
assert(same(R.DIALECT.validate(regexp(repeat((u: any) => u.missing((m: any) => m.kind("atom")), 0n))), ["element: missing atom"]));
raises(ValueError, () => RText.ToText(regexp(seq(a, (u: any) => u.error((e: any) => e)))), "Python RegExp error has no text to print");
raises(ValueError, () => PV.OfAny(regexp((u: any) => u.missing((m: any) => m))), "Python RegExp missing is not evaluated");

<a id="PYR-06"></a>
## PYR-06 · Its editions as data: Python 3.0 to 3.15, each with its own kinds it has, `atomicgroup` from 3.11

In [ ]:
// Its editions, as data: its define covers Python 3.0 to 3.15, owned by their published names, each with its own kinds
// it has (`atomicgroup` since Python 3.11, as `re`'s parser first wrote ATOMIC_GROUP; every other since 3.0) and none
// of the kinds it reads; run again, it writes nothing.
const regexp_ = store.Instances.Expressions.Python.RegExp;
const VERSIONS = Array.from({ length: 16 }, (_, minor) => `Python 3.${minor}`);
assert(same([...FS.standards(regexp_).keys()], VERSIONS) && equal(R.SINCE, new Map(R.TAGS.map((tag) => [tag, tag === "atomicgroup" ? 11 : 0]))));
assert(same(FS.tags(FS.find(regexp_, "Python 3.11")!), ["Python 3.11"]) && FS.find(regexp_, "Python 2.7") === null);
assert(equal(FS.properties(FS.find(regexp_, "Python 3.0")!), map({ bodies: ["org.python"], revision: ["version", "3.0"] })));
const older = R.TAGS.filter((tag) => tag !== "atomicgroup").map((tag) => kinds.get(tag));
assert(Array.from({ length: 11 }, (_, minor) => minor).every((minor) => same(FS.kinds(FS.find(regexp_, `Python 3.${minor}`)!), older)));
assert(Array.from({ length: 5 }, (_, i) => 11 + i).every((minor) => same(FS.kinds(FS.find(regexp_, `Python 3.${minor}`)!), R.TAGS.map((tag) => kinds.get(tag)))));
assert(same([...FS.of(regexp_, kinds.get("atomicgroup")).keys()], VERSIONS.slice(11)) && same([...FS.of(regexp_, kinds.get("pattern")).keys()], VERSIONS));
assert(FS.of(regexp_, store.registered("Expressions.Framework.Error")).size === 0);
assert(store.transaction(PythonRegExp.define)[0].get("Python_3_11") === FS.find(regexp_, "Python 3.11"));
assert(same([...FS.standards(regexp_).keys()], VERSIONS)); // nothing written again

<a id="PYR-07"></a>
## PYR-07 · Printing every kind as `re`'s syntax: a quantifier's shortest bounds, a character in its spelling or escaped where it stands, a numbered backreference before a digit in a group

In [ ]:
// Printing, as `re`'s syntax writes each kind: alternatives joined by `|`; a group, a named one, an atomic one, a
// lookaround and a scoped flags around their alternatives; a quantifier's bounds in their shortest form, then `?` where
// lazy and `+` where possessive; a class, a range, a set; a character in its spelling as written, else escaped as where
// it stands; an assertion; a backreference by number or by name, a numbered one before a digit within a group of its
// own; a conditional; a global flags; a comment.
const cases: [unknown, string][] = [
  [alts(seq(a), seq(b), seq()), "a|b|"], [seq(char(46n), char(32n), char(35n), char(10n), char(0xE9n), char(0x1F600n)), "\\.\\ \\#\\né😀"],
  [seq(char(97n, "\\x61"), char(46n, "\\."), char(0x2014n, "\\N{EM DASH}")), "\\x61\\.\\N{EM DASH}"],
  [group(seq(a), seq(b)), "(?:a|b)"], [capture(seq(a)), "(a)"], [named("word", seq(a)), "(?P<word>a)"],
  [atomic(seq(a, b)), "(?>ab)"],
  [repeat(a, 0n), "a*"], [repeat(a, 1n), "a+"], [repeat(a, 0n, 1n), "a?"], [repeat(a, 2n, 2n), "a{2}"], [repeat(a, 2n), "a{2,}"],
  [repeat(a, 2n, 5n), "a{2,5}"], [repeat(a, 0n, 5n), "a{0,5}"], [repeat(a, 0n, null, { greedy: false }), "a*?"],
  [repeat(a, 1n, 2n, { greedy: false }), "a{1,2}?"], [repeat(a, 1n, null, { possessive: true }), "a++"], [repeat(a, 0n, 1n, { greedy: true }), "a?"],
  [repeat(group(seq(a, b)), 0n, null, { possessive: false }), "(?:ab)*"],
  [klassNot(char(93n), char(45n), span(a, char(122n)), charset("word")), "[^\\]\\-a-z\\w]"],
  [klass(char(94n), char(46n), char(32n), char(8n, "\\b")), "[\\^. \\b]"], [klass(span(char(48n, "\\x30"), char(57n))), "[\\x30-9]"],
  [charset("any"), "."], [charset("digit", true), "\\D"], [charset("space"), "\\s"], [charset("word", true), "\\W"],
  [check("start"), "^"], [check("end"), "$"], [check("stringstart"), "\\A"], [check("stringend"), "\\Z"],
  [check("word"), "\\b"], [checkNot("word"), "\\B"], [check("lookahead", seq(a)), "(?=a)"],
  [checkNot("lookahead", seq(a)), "(?!a)"], [check("lookbehind", seq(a)), "(?<=a)"],
  [checkNot("lookbehind", seq(a), seq(b)), "(?<!a|b)"],
  [seq(capture(seq(a)), ref(1n)), "(a)\\1"], [seq(named("x", seq(a)), ref("x")), "(?P<x>a)(?P=x)"],
  [seq(capture(seq(a)), ref(1n), char(48n)), "(a)(?:\\1)0"], [seq(capture(seq(a)), ref(1n), repeat(char(48n), 0n)), "(a)(?:\\1)0*"],
  [seq(capture(seq(a)), ref(1n), char(48n, "\\x30")), "(a)\\1\\x30"], [seq(named("x", seq(a)), ref("x"), char(48n)), "(?P<x>a)(?P=x)0"],
  [seq(repeat(capture(seq(a)), 0n, 1n), cond(1n, seq(b), seq(a))), "(a)?(?(1)b|a)"],
  [seq(repeat(named("x", seq(a)), 0n, 1n), cond("x", seq(b))), "(?P<x>a)?(?(x)b)"],
  [seq(flags("i"), a), "(?i)a"], [flags("i", "m", seq(a)), "(?i-m:a)"], [flags("", "s", seq(a)), "(?-s:a)"],
  [flags("ax", "", seq(a), seq()), "(?ax:a|)"], [seq(comment("note"), a), "(?#note)a"], [comment(), "(?#)"],
];
for (const [spec, source] of cases) {
  assert(RText.ToText(regexp(spec)) === source, `${source} ${RText.ToText(regexp(spec))}`);
  assert(R.DIALECT.validate(regexp(spec)).length === 0, source);
}
assert(same([...new Set(cases.flatMap(([spec]) => [...F.walk(regexp(spec))].map((node) => F.kindOf(node)[0])))].sort(), [...R.TAGS].sort()));
assert(same([...RText.TEXTS.keys()].sort(), [...R.TAGS].sort()) && RText.CLOSED === 17);
assert(RText.ToText(regexp(span(char(93n), char(94n)))) === "\\]-\\^"); // a range alone, its ends spelled as within a class

<a id="PYR-08"></a>
## PYR-08 · Evaluation refuses each of its kinds by the RegExp dialect's name

In [ ]:
// Evaluation (Q11): Python3's evaluator evaluates none of its kinds, and refuses each by the RegExp dialect's name; a
// regular expression is matched by `re`, a library call, not evaluated as an expression.
for (const [spec, tag] of [[alts(seq(a)), "pattern"], [seq(a), "alternative"], [group(seq(a)), "group"], [capture(seq(a)), "capturinggroup"],
  [repeat(a, 0n), "quantifier"], [klass(a), "characterclass"], [span(a, b), "characterclassrange"],
  [charset("any"), "characterset"], [a, "character"], [check("start"), "assertion"], [ref(1n), "backreference"],
  [cond(1n, seq(a)), "conditional"], [atomic(seq(a)), "atomicgroup"], [flags("i"), "flags"], [comment("c"), "comment"]] as [unknown, string][]) {
  raises(NotImplementedError, () => PV.OfAny(regexp(spec)), `Python RegExp ${tag} is not evaluated`);
}

<a id="PYR-09"></a>
## PYR-09 · Completeness: each code `re`'s parser writes, 3.0 to 3.15, kept as data, is read as its kinds or listed as the compiler's, and each of its kinds is one of them; checked against the running Python's `re`

In [ ]:
// Completeness (Q9): `grammars/python3-re.json` lists the codes `re`'s parser writes, its operation, position and
// category codes and its flags, 3.0 to 3.15, each read as kinds of the RegExp dialect, with the attributes it gives
// them, or listed as not an expression, the compiler's, with its reason; every kind is one of them's, but those no code
// writes, each listed with its reason (the Python suite also checks it against the running Python's `re`).
const grammar = globalThis.JSON.parse(readFileSync("../../grammars/python3-re.json", "utf8"));
assert(same(Object.keys(grammar), ["grammar", "checked", "codes", "unwritten"]));
const CLASSES = ["OPCODES", "ATCODES", "CHCODES", "FLAGS"];
const ATTRIBUTES = new Map<string, readonly unknown[]>([["kind", [...R.ASSERTIONS, ...R.SETS]], ["negate", [true, false]], ["greedy", [false]],
  ["possessive", [true]], ["letter", [...R.FLAG_LETTERS]]]);
const minor = (text: string): number => Number(text.split(".")[1]);
const mapped = new Set<string>();
for (const code of grammar.codes) { // each read as kinds, with attributes, or not an expression, with its reason
  assert(CLASSES.includes(code.class) && Array.from({ length: 16 }, (_, m) => `3.${m}`).includes(code.since), code.code);
  assert(!("until" in code) || (minor(code.since) <= minor(code.until) && minor(code.until) < 15), code.code);
  if ("kinds" in code) {
    assert(!("not" in code) && code.kinds.length > 0 && code.kinds.every((kind: string) => R.TAGS.includes(kind)), code.code);
    assert(Object.entries(code.attributes ?? {}).every(([name, value]) => ATTRIBUTES.get(name)!.includes(value)), code.code);
    for (const kind of code.kinds) mapped.add(kind);
  } else {
    assert(typeof code.not === "string" && code.not !== "" && !("attributes" in code), code.code);
  }
}
const unwritten = new Map<string, string>(grammar.unwritten.map((entry: any) => [entry.kind, entry.reason]));
assert(same([...unwritten.keys()].sort(), ["comment", "pattern"]) && ![...unwritten.keys()].some((kind) => mapped.has(kind))
  && same([...new Set([...mapped, ...unwritten.keys()])].sort(), [...R.TAGS].sort()));
const read = (name: string): Record<string, unknown>[] => grammar.codes.filter((c: any) => (c.kinds ?? []).includes(name)).map((c: any) => c.attributes ?? {});
const valuesOf = (name: string, attribute: string): string[] => [...new Set(read(name).filter((x) => attribute in x).map((x) => x[attribute] as string))].sort();
assert(same(valuesOf("assertion", "kind"), R.ASSERTIONS.filter((kind) => !R.LOOKAROUNDS.includes(kind)).sort()));
assert(same(valuesOf("characterset", "kind"), [...R.SETS].sort()));
assert(same(valuesOf("flags", "letter"), [...R.FLAG_LETTERS].sort()));
assert(same(grammar.codes.filter((c: any) => (c.kinds ?? []).includes("quantifier")).map((c: any) => c.code).sort(), ["MAX_REPEAT", "MIN_REPEAT", "POSSESSIVE_REPEAT"]));
assert(equal(new Map(grammar.codes.filter((c: any) => (c.kinds ?? []).includes("atomicgroup")).map((c: any) => [c.code, c.since])), map({ ATOMIC_GROUP: "3.11" })));
assert(grammar.codes.length === 86 && mapped.size === 13);